In [1]:
import torch
import pandas as pd
from tqdm import tqdm
import traceback
import json
from src.extractors.extractor import (
    Extractor,
    BielikExtractor,
    DummyExtractor,
    PllumExtractor,
)

torch.cuda.empty_cache()

In [ ]:
DEBUG = True
model_name = "speakleash/Bielik-11B-v2.2-Instruct"

In [ ]:
acceptable_models = [
    "speakleash/Bielik-11B-v2.2-Instruct",
    "dummy",
    "CYFRAGOVPL/PLLuM-12B-nc-instruct",
]
assert (
    model_name in acceptable_models
), f"Model {model_name} not in acceptable models: {acceptable_models}"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

extractors_map = {
    "speakleash/Bielik-11B-v2.2-Instruct": BielikExtractor,
    "dummy": DummyExtractor,
    "CYFRAGOVPL/PLLuM-12B-nc-instruct": PllumExtractor,
}

extractor: Extractor = extractors_map[model_name](device=device)

In [4]:
model_size = extractor.get_memory_footprint()
print(f"Model size: {model_size:.2f} GB")

Model size: 20.80 GB


In [5]:
relations_schema = pd.read_csv("data/edc_baseline/schema.csv", header=None)

relations_description = ""
for i in range(0, len(relations_schema)):
    relation_name, relation_description = relations_schema.iloc[i]
    relations_description += f"{relation_name} - {relation_description}\n"

In [6]:
with open("data/edc_baseline/oie_few_shot_examples.txt", "r") as file:
    fewshot_examples = "".join(file.readlines())

In [7]:
with open("data/edc_baseline/dataset.txt", "r") as file:
    dataset = file.readlines()

In [8]:
def create_system_content(relations_description, fewshot_examples):
    return f"""Twoim zadaniem jest wyciągnąć jedną relację łączącą dwa obiekty występujące w tekście, jako trójkę. Trójka musi być w postaci [[Poprzednik, Relacja, Następnik]]. Poprzednik i Następnik są wyrażeniami zapisanymi w tekście. Relacja jest krótkim zapisem związku, jaki łączy Poprzednik i Następnik.
W swojej odpowiedzi przedstaw dokładnie jedną trójkę. Jeśli w tekście jest więcej możliwych trójek, wybierz najardziej prawdopodobną. Nie podawaj żadnych innych informacji czy wyjaśnień.
            
Jedyne relacje, jakie możesz wyciągnąć, to:
{relations_description}

Poniżej przykłady zdań, w których występują obiekty, dla których należy wyciągnąć trójkę:
{fewshot_examples}"""

In [9]:
def create_prompt_content(sample):
    return f"""Tekst: {sample}
Trójka:"""

In [13]:
responses = []
errors = []

system_content = create_system_content(relations_description, fewshot_examples)

for sample in tqdm(dataset[:10] if DEBUG else dataset):
    try:
        print("Sample:", sample.strip())

        template = extractor.create_messages_template(
            system_content, create_prompt_content(sample)
        )

        response = extractor.get_response_text(template)
        responses.append(response)
        print("Response:", response)
    except Exception as e:
        tb = traceback.format_exc()
        error_message = f"Error for sample: {sample.strip()}\n{tb}\n"
        errors.append(error_message)
        responses.append("Error\n")
        print("Error:", error_message)

with open("data/edc_baseline/extracted_relations.json", "w", encoding="utf-8") as file:
    json.dump(dict(responses=responses), file, ensure_ascii=False, indent=2)

with open("data/edc_baseline/errors.json", "w", encoding="utf-8") as file:
    json.dump(dict(errors=errors), file, ensure_ascii=False, indent=2)

  0%|          | 0/50 [00:00<?, ?it/s]

Sample: 25 stycznia 1973), znany również jako El Angelito – amerykański perkusista, muzyk sesyjny portorykańskiego pochodzenia. Tony Laureano znany jest przede wszystkim z występów w deathmetalowej grupie muzycznej Nile, której był członkiem w latach 2000-2004.


  2%|▏         | 1/50 [00:02<01:51,  2.27s/it]

Response: [['Tony Laureano', 'członek', 'Nile']]
Sample: Abdellah El-Ajri znany bardziej jako Abdellah Blinda (ur. Jako zawodnik grał w FUS Rabat.


  4%|▍         | 2/50 [00:03<01:30,  1.88s/it]

...

Response: [['Lyndhurst', 'lokalizacja', 'Nowa Południowa Walia']]
Sample: Maltby – osada w Anglii, w hrabstwie Lincolnshire, w dystrykcie East Lindsey.


 98%|█████████▊| 49/50 [01:19<00:01,  1.51s/it]

Response: [['Maltby', 'lokalizacja', 'Anglia']]
Sample: Sławek Jaskułke, właśc. 2 stycznia 1979 w Pucku) − polski pianista jazzowy, kompozytor muzyki fortepianowej, orkiestrowej, teatralnej i filmowej, aranżer, producent muzyczny.


100%|██████████| 50/50 [01:21<00:00,  1.62s/it]

Response: [['Sławek Jaskułke', 'zawód', 'Pianista jazzowy']]
